# Week 2 · Transformer 基础

> **本周一句话**:从一个不带任何"注意力"的 Bigram 查表 baseline 开始,逐步加上 token/pos embedding、单头 attention、多头、FFN、残差、LayerNorm、深度堆叠,看 val loss 从 5.43 一路降到 4.14。

这是项目最厚的一周。学完之后你写的代码就是一个能 work 的 GPT。

## 0. 本周目标

4 个版本的演进:

| 版本 | 关键改动 | 参数量 | val loss | 写出的诗 |
|---|---|---|---|---|
| Bigram | 一张 (V, V) 查表,无任何上下文 | 91 M | **5.43** | 单字接龙,毫无连贯 |
| v0.1 | + token/pos embed + 单头 attention(16 维)| 0.79 M | 6.03 | **反而退步** —— 容量瓶颈 |
| v0.2 | + 多头(4 头)+ FFN + LayerNorm + 残差 | 2.67 M | 5.10 | 击穿 Bigram,初见句式 |
| v0.3 | + 6 层 Block + GPT-2 风格初始化 | 6.37 M | **4.14** | 开始押韵、有意境 |

**注意 v0.1 的"反而退步"** —— 这是教学上故意的失败案例。比起直接给你成功的代码,看到一次失败 + 知道为什么失败,理解会深得多。

## 1. 前置知识

**必备**:
- Week 1 跑通(数据 + tokenizer + get_batch)
- 矩阵乘法的形状变换:`(A, B) @ (B, C) = (A, C)`
- 知道什么是"梯度下降"、"反向传播"、"交叉熵"

**这周第一次遇到的话**:
- `nn.Embedding`、`nn.Linear`、`nn.LayerNorm`、`nn.Dropout`、`F.softmax`、`F.cross_entropy`
- `register_buffer` vs `nn.Parameter` 的区别(在 6.2 节讲)

## 2. 核心概念

### 2.1 Bigram 是什么

最朴素的语言模型:**只看上一个字预测下一个字**,完全忽略更远的上下文。

实现就一张 `nn.Embedding(vocab_size, vocab_size)` 表 —— 输入是 token id,直接查出来一行 `vocab_size` 维向量当 logits。

```python
class BigramModel(nn.Module):
    def __init__(self, V):
        super().__init__()
        self.lut = nn.Embedding(V, V)   # 91M for V=9563
    def forward(self, x):
        return self.lut(x)              # (B, T, V)
```

**为什么 baseline 重要**?它给后面所有模型一条"如果你比我还差,说明你白搭了 attention"的底线。Bigram 的 val 5.43 就是这条线。

### 2.2 单头 Self-Attention(v0.1 的失败)

加上位置编码 + 单头 attention 之后,模型理论上能"看上文"了。但我们故意把 head_size 设成 16(很小),结果 val loss 是 **6.03 比 Bigram 还差**。

**为什么会更差**?

- Bigram 直接用 `(V, V) = (9563, 9563)` 的大表,能"死记硬背"所有 bigram pattern
- v0.1 的 attention 只有 16 维信息瓶颈,所有上下文压成 16 个数,**还不如直接查表**

**教学价值**:看到一个反例,理解"加了 attention ≠ 必然变好",要看容量配不配。

**Attention 的数学公式**(逃不开):

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

`Q / K / V` 都是同一个输入 `x` 经过 3 个独立 `Linear` 投影出来的(self-attention 的 "self" 就这个意思)。

In [ ]:
# 一头 self-attention 的最小实现(对照 model/minigpt_v01.py:SingleHead)
import torch
import torch.nn as nn
import torch.nn.functional as F

B, T, C = 2, 4, 8     # batch=2, seq=4, embed=8
head_size = 4
x = torch.randn(B, T, C)

Wq = nn.Linear(C, head_size, bias=False)
Wk = nn.Linear(C, head_size, bias=False)
Wv = nn.Linear(C, head_size, bias=False)

q, k, v = Wq(x), Wk(x), Wv(x)         # 各 (B, T, head_size)
scores = q @ k.transpose(-2, -1) * head_size ** -0.5   # (B, T, T)

tril = torch.tril(torch.ones(T, T))
scores = scores.masked_fill(tril == 0, float("-inf"))  # causal mask
weights = F.softmax(scores, dim=-1)
out = weights @ v                      # (B, T, head_size)

print(f"weights[0]:\n{weights[0]}")
print(f"\n注意上三角全是 0(被 -inf softmax 压成 0)—— 这就是 causal mask")

### 2.3 Causal mask 为什么是 -inf 而不是 0

softmax 公式 `exp(x) / sum(exp(x))`:
- 如果填 `0`,`exp(0) = 1` —— 仍然分到权重,模型偷看未来
- 如果填 `-inf`,`exp(-inf) = 0` —— 权重严格 0

只有 `-inf` 才能彻底"屏蔽"未来位置。

**上三角还是下三角**?`scores[i][j]` 表示位置 i 看位置 j。
- 列号 > 行号(上三角)= "看未来" → 屏蔽
- 列号 ≤ 行号(下三角)= "看当前及过去" → 保留

### 2.4 为什么 scores 要除 √d_k

不除会怎样?`q @ k.T` 的结果方差和 `d_k` 成正比,`d_k` 大了之后 scores 会变得很大。

softmax 输入数字之间差距大时,**会一边倒**(一个位置吃 99% 权重,其他全是 0):
- 信息瓶颈:模型只看一个位置
- 梯度消失:softmax 接近 one-hot 时对输入的梯度接近 0,学不动

除 √d_k 把 scores 压回标准正态范围,让 softmax 输出平滑可学。

### 2.5 多头 + FFN + 残差 + LayerNorm(v0.2 击穿 Bigram)

v0.2 一次性加了 4 个东西:

1. **多头**:n_embed=128 切成 4 个头,每头 32 维。每个头学一种不同的"看上文方式"(押韵 / 对仗 / 意境...)。一头比四头是单点失败,4 头并联鲁棒得多

2. **FFN**:每个位置过一个 `Linear(C, 4C) → GELU → Linear(4C, C)`。中间扩展 4 倍是经验值。**attention 让位置之间交换信息;FFN 让每个位置内部做非线性加工**

3. **残差连接** `x = x + sublayer(x)`:让原始输入有一条"直通车"绕过子层。深层网络能训得动的根本原因。改成 `x = sublayer(x)` 会**梯度消失**,深层学不到东西

4. **LayerNorm**:对每个 token 的 C 维向量做归一化,让数值分布稳定。**我们用 Pre-LN**(`x + Sublayer(LN(x))`),GPT-2 之后的标准做法,比 Post-LN(原 Transformer 论文)更稳定

一个完整 Block 长这样:

```python
class Block(nn.Module):
    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # 残差 + Pre-LN + Attention
        x = x + self.ffn(self.ln2(x))    # 残差 + Pre-LN + FFN
        return x
```

### 2.6 多层堆叠 + 权重初始化(v0.3 的关键细节)

把上面的 Block 堆 6 层就是 v0.3。但**直接堆深会训不动** —— 必须加 GPT-2 风格的初始化:

```python
def _init_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

self.apply(_init_weights)
```

**为什么需要**?PyTorch 默认初始化(Kaiming)是为单层 CNN 设计的。6 层叠起来后,每层方差会累积放大或缩小,初始 forward 时数值就跑偏了 —— 6 层后 logits 范围可能爆到 1e3 或缩到 1e-3,loss 完全乱。

std=0.02 是 GPT-2 实验出来的"适合深层 Transformer"的方差。**判断是否合理的信号**:初始 loss 应该接近 `log(vocab_size)`(纯随机猜测的理论值)。v0.3 初始 loss 应该在 **9.16** 附近(`log(9563) ≈ 9.17`),偏离太多说明初始化没生效。

## 3. 代码地图

| 文件 | 行数 | 干什么 |
|---|---|---|
| `model/bigram.py` | 40 | Bigram baseline |
| `model/minigpt_v01.py` | 80 | 单头 `SingleHead` + `MiniGPTv01`(自包含教学版) |
| `model/layers.py` | 75 | 共享:`MultiHeadAttention` / `FeedForward` / `Block` |
| `model/minigpt_v02.py` | 60 | 单 Block 版本,用 `layers.Block` |
| `model/minigpt_v03.py` | 80 | n 层 stack + GPT-2 初始化 —— **后续 Week 5/6/7/8 都用它** |
| `train/train_bigram.py` | 35 | 训练 Bigram |
| `train/train_v01.py` 至 `train_v03.py` | 各 ~40 | 对应 4 个版本的训练入口 |
| `train/utils.py` | 90 | `get_batch / estimate_loss / train_loop` |
| `inference/generate.py` | 60 | CLI 生成,支持所有 4 个 baseline |

## 4. 动手做

按顺序跑 4 个训练,**每跑完一个对一下 val loss**,差距太大就回去找原因。

In [ ]:
import subprocess, sys


def run(cmd):
    """跑子进程并把输出实时打印到 cell。
    subprocess.run 默认把子进程 stdout 写到 kernel 原始 fd, notebook 看不到;
    用 Popen 逐行回读才能在 cell 里实时看到脚本的 print。
    "python" 换成 sys.executable, 确保用当前 kernel 解释器。"""
    cmd = [sys.executable if c == "python" else c for c in cmd]
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise SystemExit(f"{cmd} 退出码 {proc.returncode}")

run(["python", "../train/train_bigram.py"])
# 预期: val loss 收敛到 ~5.43,用时 ~30 秒(T4)

In [ ]:
run(["python", "../train/train_v01.py"])
# 预期: val loss 6.03 左右,比 Bigram 差! ~1 分钟
# 故意的失败案例,看到这个数字就对了

In [ ]:
run(["python", "../train/train_v02.py"])
# 预期: val loss ~5.10,击穿 Bigram baseline,~2 分钟

In [ ]:
run(["python", "../train/train_v03.py"])
# 预期: val loss ~4.14,本周最终模型,~5 分钟

In [ ]:
# 让 v0.3 写一首诗
run(["python", "../inference/generate.py",
                "--model", "v03", "--start", "月", "--tokens", "150"])

## 5. 自测题

本周的自测题在独立文件里:**`courses/week2_self_test.md`** 共 8 大类 30 道,题目+答案都有。建议:

1. 先盖住答案,扫一遍题目
2. 心里给每题打分:清楚 / 大概懂 / 完全没思路
3. 把"完全没思路"的回去看对应小节

**不要直接看答案**。教育心理学叫 "desirable difficulty" —— 大脑只有在尝试调用并失败时,才会真正记住新输入的信息。

## 6. 容易踩的坑

**坑 1:Bigram 的"虚高参数量"**

Bigram 91M 参数看着唬人,但都是死记的查找表,毫无泛化能力。**参数量 ≠ 能力**,看 val loss 才数。

**坑 2:v0.1 单头 16 维的容量瓶颈**

如果你把 v0.1 的 head_size 改到 64 甚至 128,val loss 会比 Bigram 好(Karpathy 的 nanoGPT 就用了 64)。我们故意用 16 是为了制造对比。

**坑 3:v0.3 没加 `_init_weights` → loss 不收敛 / 收敛慢**

index.ipynb 早期版本的 v0.3 就因为没初始化导致深层学不动。修复后 val 才能下到 4.14。

**坑 4:`generate()` 忘了切 `eval()` → 输出含 Dropout 噪声**

推理时如果还开着 Dropout,采样会变得很随机(实际效果像温度被偷偷调高)。我们的 `generate()` 都显式 `self.eval()` / `self.train()`,**不要去掉**。

**坑 5:`generate` 不截 `idx[:, -self.block_size:]` → 上下文超 128 崩**

`position_embedding` 只学了 0~127 这 128 个位置。超过会 `IndexError` 或拿到错误位置向量。我们的实现都加了截断,你自己改的时候别忘。

## 7. 进入 Week 3 前

现在你应该:
- ☑ 4 个 train_*.py 都跑过,看到 val loss 的下降阶梯
- ☑ 能不看代码画出 Q/K/V → scores → mask → softmax → V 的数据流图
- ☑ 能解释为什么 v0.1 比 Bigram 还差
- ☑ 能解释 Pre-LN vs Post-LN 的区别

Week 3 我们结构基本不变,但要给训练循环加 4 样东西(LR schedule / AMP / grad clip / weight decay),让训练变得"工业级健康"。